# PB04 — Rest EEG come Baseline di Normalizzazione

**Obiettivo**: usare le statistiche del rest per normalizzare le epoche img/read,
riducendo la variabilità inter-soggetto senza richiedere dati labeled.

**Metodo**: per ogni soggetto, calcola media e std del rest → z-score delle epoche img.
Confronta bacc prima/dopo la normalizzazione su un classificatore baseline.

In [5]:
from pathlib import Path
import json

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT = project_root / 'data' / 'raw_csv' / 'training_set'
CONFIGS   = project_root / 'configs' / 'label_schemes'
SFREQ = 256
N_CHAN = 61
N_SAMP = 384
CLUSTER_SCHEME = 'concr4'

with open(CONFIGS / 'label2idx.json') as f:
    word2idx = json.load(f)
with open(CONFIGS / f'labelid2cluster_{CLUSTER_SCHEME}.json') as f:
    label2cluster = {int(k): v for k, v in json.load(f).items()}
N_CLASSES = len(set(label2cluster.values()))

print(f'Schema: {CLUSTER_SCHEME} | {N_CLASSES} classi | DATA_ROOT: {DATA_ROOT}')

Schema: concr4 | 4 classi | DATA_ROOT: /home/daniele_u/miralis-hypergraph-imagined-speech/data/raw_csv/training_set


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
print('Import OK')

Import OK


In [7]:
# ============================================================
# FUNZIONI
# ============================================================

def load_rest_stats(subj_id: int, data_root: Path):
    """Calcola media e std del rest per soggetto → (mean, std) shape (N_CHAN, N_SAMP)."""
    epochs = []
    for sess_dir in sorted(data_root.glob(f'P{subj_id:03d}_S*')):
        for f in sorted(sess_dir.glob('riposo_*.csv')):
            x = pd.read_csv(f, header=None).values.astype(np.float32)
            if x.shape == (N_CHAN, N_SAMP):
                epochs.append(x)
    if not epochs:
        return None, None
    arr = np.array(epochs)  # (n, 61, 384)
    return arr.mean(axis=0), arr.std(axis=0) + 1e-8  # evita divisione per zero


def load_img_epochs(subj_id: int, data_root: Path):
    """Carica epoche img con relative label cluster."""
    X, y = [], []
    for sess_dir in sorted(data_root.glob(f'P{subj_id:03d}_S*')):
        for f in sorted(sess_dir.glob('*_img.csv')):
            word = f.stem.replace('_img', '')
            if word not in word2idx:
                continue
            lid = word2idx[word]
            if lid not in label2cluster:
                continue
            x = pd.read_csv(f, header=None).values.astype(np.float32)
            if x.shape == (N_CHAN, N_SAMP):
                X.append(x)
                y.append(label2cluster[lid])
    return np.array(X), np.array(y)


def classify_subj(X, y, normalize_with=None):
    """Logistic regression con LOO-CV. Opzionalmente normalizza con rest."""
    if normalize_with is not None:
        rest_mean, rest_std = normalize_with
        X = (X - rest_mean[None]) / rest_std[None]
    X_flat = X.reshape(len(X), -1)
    scaler = StandardScaler()
    X_flat = scaler.fit_transform(X_flat)
    from sklearn.model_selection import cross_val_predict
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(C=0.01, max_iter=500, solver='lbfgs')
    y_pred = cross_val_predict(clf, X_flat, y, cv=min(5, len(y)))
    return balanced_accuracy_score(y, y_pred)


print('Funzioni OK')

Funzioni OK


In [8]:
# ============================================================
# CONFRONTO: bacc con e senza normalizzazione rest
# ============================================================

all_subj = sorted(set(
    int(d.name.split('_')[0][1:]) for d in DATA_ROOT.iterdir() if d.is_dir()
))[:20]  # usa i primi 20 per ora

results = []
for sid in tqdm(all_subj, desc='Soggetti'):
    X, y = load_img_epochs(sid, DATA_ROOT)
    if len(X) < 20:
        continue
    rest_mean, rest_std = load_rest_stats(sid, DATA_ROOT)

    bacc_base = classify_subj(X, y, normalize_with=None)
    bacc_rest = classify_subj(X, y, normalize_with=(rest_mean, rest_std)) if rest_mean is not None else np.nan

    results.append({'subj': sid, 'bacc_baseline': bacc_base, 'bacc_rest_norm': bacc_rest})
    print(f'P{sid:03d}: baseline={bacc_base:.3f}  rest_norm={bacc_rest:.3f}  Δ={bacc_rest-bacc_base:+.3f}')

df = pd.DataFrame(results)
print(f'\nMedia baseline: {df["bacc_baseline"].mean():.3f}')
print(f'Media rest_norm: {df["bacc_rest_norm"].mean():.3f}')
print(f'Δ medio: {(df["bacc_rest_norm"] - df["bacc_baseline"]).mean():+.3f}')

Soggetti:   0%|          | 0/20 [00:00<?, ?it/s]

P000: baseline=0.270  rest_norm=0.270  Δ=+0.000
P001: baseline=0.239  rest_norm=0.239  Δ=+0.000


KeyboardInterrupt: 

In [ ]:
# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(df))
w = 0.35
ax.bar(x - w/2, df['bacc_baseline'], w, label='Baseline', color='steelblue', alpha=0.8)
ax.bar(x + w/2, df['bacc_rest_norm'], w, label='Rest normalization', color='darkorange', alpha=0.8)
ax.axhline(1/N_CLASSES, color='red', linestyle='--', label=f'Chance ({1/N_CLASSES:.2f})')
ax.set_xticks(x)
ax.set_xticklabels([f'P{r["subj"]:03d}' for _, r in df.iterrows()], rotation=45, ha='right')
ax.set_ylabel('bacc')
ax.set_title('Effetto normalizzazione rest su classificazione imagined speech')
ax.legend()
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB04_rest_normalization_effect.png', dpi=150)
plt.show()